# Qwen3-8B Task 2 Hybrid V2 experiment
Enable GPU T4 x2 and Internet. Run fold 0 first and report its JSON before running the remaining folds or full training.

In [ ]:
!pip install -q -U "transformers>=4.51" "peft>=0.15" accelerate bitsandbytes openpyxl scikit-learn
from pathlib import Path
import zipfile
scripts = list(Path('/kaggle/input').rglob('kaggle_factor_qwen.py'))
if not scripts:
    archives = list(Path('/kaggle/input').rglob('kaggle_factor_qwen_v2.zip'))
    assert len(archives) == 1, archives
    extracted = Path('/kaggle/working/task2_qwen_package')
    with zipfile.ZipFile(archives[0]) as z: z.extractall(extracted)
    scripts = list(extracted.rglob('kaggle_factor_qwen.py'))
assert len(scripts) == 1, scripts
script = scripts[0]
print(script)

## Fast package and split check

In [ ]:
!python -u {script} --stage preflight

## Strict OOF fold (start with FOLD=0)

In [ ]:
import subprocess
RUN_FOLD = True
FOLD = 0
if RUN_FOLD:
    subprocess.run(['python', '-u', str(script), '--stage', 'fold', '--fold', str(FOLD), '--epochs', '2'], check=True)

## Run only after all five fold NPZ files are present or attached

In [ ]:
RUN_SUMMARY = False  # set True only when all five fold NPZ files are attached/present
if RUN_SUMMARY:
    subprocess.run(['python', '-u', str(script), '--stage', 'summarize-oof'], check=True)

## Full-data training (only after OOF is promising)

In [ ]:
RUN_FULL = False  # set True only after the OOF summary is promising
if RUN_FULL:
    subprocess.run(['python', '-u', str(script), '--stage', 'full', '--epochs', '2', '--save-adapter'], check=True)

In [ ]:
import zipfile
MAKE_FULL_ARCHIVE = False
work = Path('/kaggle/working')
archive = work / 'qwen3-8b-factor-v2_full_prediction.zip'
if MAKE_FULL_ARCHIVE:
  with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('qwen3-8b-factor-v2_test_probabilities.npz', 'qwen3-8b-factor-v2_full_results.json', 'qwen3-8b-factor-v2_oof_summary.json', 'qwen3-8b-factor-v2_oof_probabilities.npz'):
        path = work / name
        if path.exists(): z.write(path, path.name)
    adapter = work / 'qwen3-8b-factor-v2_full_adapter'
    if adapter.exists():
        for path in adapter.rglob('*'):
            if path.is_file(): z.write(path, path.relative_to(work))
  print(archive)